# === USER CONFIGURATION ===

In [ ]:
# # === USER CONFIGURATION ===
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ CAREFUL, please read the README_Notebooks.md file before
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

WORKDIR = "PATH/TO/WORKDIR"  # Working directory for sen2vm processing

# Path to downloaded product
PATH_L1B_DATA = "PATH/TO/L1B_PRODUCT"

# Path to GIPP directory
PATH_GIPP = "PATH/TO/GIPP"
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ If you do not have your own GIPP folder, you may use the inputs-download-notebook to download it, then you may indicate the path were you 
# /!\/!\/!\ downloaded it here in PATH_GIPP
# /!\/!\/!\ If you have your own GIPP folder, please note that this current notebook will search for a subfolder with mission S2[A/B/C] inside the GIPP folder
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

# Path to DEM directory 
PATH_DEM = "PATH/TO/DEM"

# Path to GEOID directory
PATH_GEOID = "PATH/TO/GEOID"
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ If you do not have your own GEOID folder, you may use the inputs-download-notebook to download it, then you may indicate the path were you
# /!\/!\/!\ downloaded it here in  
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

# Path to IERS file
PATH_IERS = "PATH/TO/IERS_FILE"
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\
# /!\/!\/!\ If you do not have your own IERS, you may let this path empy (i.e. "") and execute the cell number 2 named #IERS Download
# /!\/!\/!\ In this case, the IERS will be downloaded in the WORKDIR
# /!\/!\/!\/!\/!\/!\/!\/!\/!\/!\/!\

# The output folder is put in the WORKDIR
OUTPUT_FOLDER = WORKDIR + "/output/DIRECT_gdal"

# === SEN2VM OPTIONS ===
UTM_EPSG = 32628 # UTM zone EPSG code for the ROI
LOCATION = {
    "ul_x": 283910,
    "ul_y": 3641660,
    "lr_x": 354280,
    "lr_y": 3608316.0
}

# Grid step in pixels for sen2vm grid generation 
# These values represent the grid step in pixels
# In operation, this step is linked to DEM step which is not constant over the world, pending the Latitude
STEPS = {
    "10m_bands": 4.5,  # Around 45m in operation (pending laittude) with 30m and also 90m DEM
    "20m_bands": 4.5,  # Around 90m in operation (pending laittude) with 30m and also 90m DEM
    "60m_bands": 3.0   # Around 180m in operation (pending laittude) with 30m and also 90m DEM
}

# === GDAL ORTHO OPTIONS ===

ORTHO_SETTINGS = {
    "keep_bands": ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B10", "B11", "B12"],  # list of bands to keep for orthorectification
    "keep_detectors": ["07","08","09","10","11"]  # list of the detectors to keep 
}

# === Docker Options ===
# Building Docker images may take several minutes.
# By setting REMOVE_DOCKER_IMAGE = False, the images are kept,
# which significantly speeds up subsequent executions.
#
# However, Docker images use disk space (~4 GB in this case).
#
# Docker images can be manually removed using:
#   docker images
#   docker rmi <image_id>
#
REMOVE_DOCKER_IMAGE = False  # True = remove Docker images after execution

# IERS Download

In [ ]:
# === DOWNLOAD IERS ===
import os
import re
import requests
import sys
from datetime import datetime

if len(PATH_IERS) != 0: # Safety in case the PATH_IERS isn't empty
    print("Stopping because PATH_IERS is not empty")
    sys.exit(0)

if not os.path.exists(WORKDIR):
    raise RuntimeError(f"WORKDIR directory not found: {WORKDIR}")

print("Bulletin output directory: ", WORKDIR)

PATH_IERS = WORKDIR # then our file will be there
# =====================================================================
# REMOVE EXISTING IERS BULLETINS
# =====================================================================

for f in os.listdir(WORKDIR):
    if f.startswith("bulletina-") and f.endswith(".txt"):
        os.remove(os.path.join(WORKDIR, f))
        print("Removed old bulletin A: ", f)

    if f.startswith("bulletinb-") and f.endswith(".txt"):
        os.remove(os.path.join(WORKDIR, f))
        print("Removed old bulletin B: ", f)

print("Cleanup of old bulletins complete.\n")
# =====================================================================
# EXTRACT PRODUCT DATE (FROM DATASTRIP)
# =====================================================================

datastrip_dir = os.path.join(PATH_L1B_DATA, "DATASTRIP")

if not os.path.isdir(datastrip_dir):
    raise RuntimeError(f"DATASTRIP directory not found: {datastrip_dir}")

datastrip_entries = os.listdir(datastrip_dir)
if not datastrip_entries:
    raise RuntimeError(f"No DATASTRIP found in: {datastrip_dir}")

# Take the first DATASTRIP product
datastrip_name = datastrip_entries[0]

match = re.search(r"_S(\d{8})T\d{6}_", datastrip_name)
if not match:
    raise RuntimeError("Could not extract product date from DATASTRIP name.")

product_date_str = match.group(1)

year = int(product_date_str[:4])
month = int(product_date_str[4:6])
day = int(product_date_str[6:8])

product_date = datetime(year, month, day)

print("Product date extracted from DATASTRIP: ", product_date.date(), "\n")
# =====================================================================
# Roman conversion 
# =====================================================================

def int_to_roman(n):
    vals = [
        (1000, 'm'), (900, 'cm'), (500, 'd'), (400, 'cd'),
        (100, 'c'), (90, 'xc'), (50, 'l'), (40, 'xl'),
        (10, 'x'), (9, 'ix'), (5, 'v'), (4, 'iv'), (1, 'i')
    ]
    res = ""
    for v, s in vals:
        while n >= v:
            res += s
            n -= v
    return res
# =====================================================================
# Bulletin A
# =====================================================================

roman_year = int_to_roman(year - 1987)
doy = product_date.timetuple().tm_yday
index = (doy - 1) // 7 + 1

print(f"Bulletin A Roman year: {roman_year}")
print(f"Initial weekly index: {index}\n")

found = False

while index > 0:
    index_str = f"{index:03d}"
    url = f"https://datacenter.iers.org/data/6/bulletina-{roman_year}-{index_str}.txt"
    print("Trying bulletin: ", url)

    response = requests.get(url)

    if response.status_code == 200:
        print("Bulletin found: ", index_str)
        dest_file = os.path.join(WORKDIR, f"bulletina-{roman_year}-{index_str}.txt")
        found = True
        break

    index -= 1

if not found:
    raise RuntimeError("No Bulletin A available for current or previous weeks.")

with open(dest_file, "wb") as f:
    f.write(response.content)

print("Downloaded: ", dest_file)

# Sen2VM configuration

In [ ]:
# === GENERATE CONFIG.JSON  ===

import os
import json
import re
import shutil
from numpy import double

USERCONF_DIR = os.path.join(WORKDIR, "UserConf")
os.makedirs(USERCONF_DIR, exist_ok=True)

print("UserConf directory: ", USERCONF_DIR)
print("Geoid directory: ", PATH_GEOID)

# =====================================================
# 1. Extract mission from DATASTRIP
# =====================================================

datastrip_dir = os.path.join(PATH_L1B_DATA, "DATASTRIP")

if not os.path.isdir(datastrip_dir):
    raise RuntimeError(f"DATASTRIP directory not found: {datastrip_dir}")

datastrip_entries = os.listdir(datastrip_dir)
if not datastrip_entries:
    raise RuntimeError(f"No DATASTRIP found in: {datastrip_dir}")

datastrip_name = datastrip_entries[0]

match = re.match(r"(S2[A-C])_OPER_", datastrip_name)
if not match:
    raise RuntimeError("Cannot extract mission (S2A/S2B/S2C) from DATASTRIP name.")

mission = match.group(1)
# =====================================================
# 2. Docker paths inside /workspace
# =====================================================

docker_l1b  = "/data/L1B"
docker_dem  = "/data/DEM"
docker_gipp = f"/data/GIPP/{mission}"
# =====================================================
# 3. Geoid management
# =====================================================

# Detect .gtx inside PATH_GEOID
geoid_files = [f for f in os.listdir(PATH_GEOID) if f.lower().endswith(".gtx")]
if len(geoid_files) == 0:
    raise RuntimeError("No .gtx geoid file found in PATH_GEOID.")

docker_geoid = f"/data/GEOID/{geoid_files[0]}"
# =====================================================
# 4. Locate IERS bulletin on host
# =====================================================

iers_host = None

for f in os.listdir(PATH_IERS):
    if f.startswith("bulletin"):
        iers_host = os.path.join(PATH_IERS, f)
        break

if iers_host is None:
    raise RuntimeError("IERS bulletin not found inside PATH_IERS directory.")

docker_iers = f"/data/IERS/{os.path.basename(iers_host)}"
print(docker_iers)
# =====================================================
# 5. Build config dictionary
# =====================================================

config = {
    "l1b_product": docker_l1b,
    "gipp_folder": docker_gipp,
    "auto_gipp_selection": True,
    "grids_overwriting": True,
    "dem": docker_dem,
    "geoid": docker_geoid,
    "iers": docker_iers,
    "operation": "direct",
    "deactivate_available_refining": False,
    "steps": {
        "10m_bands": STEPS["10m_bands"],
        "20m_bands": STEPS["20m_bands"],
        "60m_bands": STEPS["60m_bands"]
    },
    "export_alt": True
}
# =====================================================
# Save config.json
# =====================================================

config_path = os.path.join(USERCONF_DIR, "config.json")

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("Configuration file generated: ")
print(config_path)

In [ ]:
# === GENERATE PARAMS.JSON ===

import os
import json
import re

USERCONF_DIR = os.path.join(WORKDIR, "UserConf")
os.makedirs(USERCONF_DIR, exist_ok=True)

print("UserConf directory: ", USERCONF_DIR)
# =====================================================
# Locate GRANULE folders
# =====================================================

GR_TARGET_DIR = os.path.join(PATH_L1B_DATA, "GRANULE")

if not os.path.exists(GR_TARGET_DIR):
    raise RuntimeError("GRANULE directory not found inside L1B SAFE.")

granule_folders = [
    os.path.join(GR_TARGET_DIR, d)
    for d in os.listdir(GR_TARGET_DIR)
    if os.path.isdir(os.path.join(GR_TARGET_DIR, d))
]

print("Found ", len(granule_folders), " granule folders.")
# =====================================================
# Extract detectors and bands from JP2
# =====================================================

detectors = set()
bands = set()

pattern = r"_D(\d+)_B(\d{1,2}[A]?)\.jp2$"

for granule in granule_folders:
    img_data_dir = os.path.join(granule, "IMG_DATA")

    if not os.path.isdir(img_data_dir):
        continue

    for fname in os.listdir(img_data_dir):
        match = re.search(pattern, fname)
        if match:
            detectors.add(match.group(1))
            bands.add(f"B{match.group(2)}")

detectors = sorted(detectors)
bands = sorted(bands)

print("Detected detectors: ", detectors)
print("Detected bands: ", bands)
# =====================================================
# Write params.json
# =====================================================

# Validate keep_detectors: must be a non-empty list (user requirement)
if not isinstance(ORTHO_SETTINGS.get("keep_detectors"), list) or len(ORTHO_SETTINGS["keep_detectors"]) == 0:
    raise RuntimeError("ORTHO_SETTINGS['keep_detectors'] must be a non-empty list of detector ids (e.g. ['01','02']).")

# Filter detectors to only include those selected by the user
keep_detectors_list = ORTHO_SETTINGS["keep_detectors"]
selected_detectors = [d for d in keep_detectors_list if d in detectors]

# Validate that all requested detectors exist
missing_detectors = [d for d in keep_detectors_list if d not in detectors]
if missing_detectors:
    raise RuntimeError(f"Some requested detectors are not available in the product: {missing_detectors}. Available detectors: {detectors}")

# Only include selected detectors and bands in params.json
params = {
    "detectors": sorted(selected_detectors),
    "bands": sorted(ORTHO_SETTINGS["keep_bands"])
}

params_path = os.path.join(USERCONF_DIR, "params.json")

with open(params_path, "w") as f:
    json.dump(params, f, indent=4)

print("params.json written to: ", params_path)
print(f"  - Detectors: {params['detectors']}")
print(f"  - Bands: {params['bands']}")

# Sen2VM run

In [ ]:
# === RUN SEN2VM (Docker: BUILD + RUN + CLEAN) ===

import os
import subprocess

# Notebook location (NOT relative to CWD)
notebook_dir = os.getcwd()

dockerfile_dir = os.path.abspath(os.path.join(
    notebook_dir,
    "..", ".."
))

config_inside = "/workspace/UserConf/config.json"
params_inside = "/workspace/UserConf/params.json"
# =====================================================
# 1. BUILD DOCKER IMAGE
# =====================================================

print(f"Building Docker image 'sen2vm' from: {dockerfile_dir}")
cmd_build = [
    "docker", "build",
    "-t", "sen2vm",
    dockerfile_dir
]

print("Command:", " ".join(cmd_build), "\n")
subprocess.run(cmd_build, check=True)
print("Docker image built successfully.\n")
# =====================================================
# 2. RUN SEN2VM CONTAINER
# =====================================================

cmd_run = [
    "docker", "run",
    "--rm",
    "-v", f"{PATH_L1B_DATA}:/data/L1B",
    "-v", f"{PATH_DEM}:/data/DEM",
    "-v", f"{PATH_GIPP}:/data/GIPP",
    "-v", f"{PATH_GEOID}:/data/GEOID",
    "-v", f"{PATH_IERS}:/data/IERS",
    "-v", f"{WORKDIR}:/workspace",
    "sen2vm",
    "-c", config_inside,
    "-p", params_inside
]

print("Running Docker container...\n")
print("Command:", " ".join(cmd_run), "\n")
subprocess.run(cmd_run, check=True)
print("\nDocker execution complete.\n")
# =====================================================
# 3. REMOVE DOCKER IMAGE
# =====================================================

if REMOVE_DOCKER_IMAGE:
    print("Removing Docker image 'sen2vm'...")
    subprocess.run(["docker", "rmi", "-f", "sen2vm"], check=True)
    print("Docker images removed.\n")
else:
    print("Docker images kept.\n")

    # Generate Orthorectification images

In [ ]:
import os
import subprocess
import glob
import json
import re

# =====================================================
# SAFETY CHECKS
# =====================================================
assert os.path.exists(PATH_L1B_DATA), "L1B path missing"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
assert os.path.exists(OUTPUT_FOLDER), "Output folder missing"

# =====================================================
# Locate product name
# =====================================================
product = os.path.basename(os.path.normpath(PATH_L1B_DATA))

# =====================================================
# Locate XML
# =====================================================
xml_list = glob.glob(
    os.path.join(
        PATH_L1B_DATA,
        "S2*_MTD_*.xml"
    )
)

if len(xml_list) == 0:
    raise RuntimeError("No DATASTRIP MTD XML found")

xml_path = xml_list[0]
xml_name = os.path.basename(xml_path)

# XML path inside docker (relative, after cd)
xml_docker = f"./{xml_name}"

print("Using XML: ", xml_path)

# =====================================================
# Read params.json to get selected bands and detectors
# =====================================================
params_path = os.path.join(WORKDIR, "UserConf", "params.json")
if not os.path.exists(params_path):
    raise RuntimeError(f"params.json not found at {params_path}")

with open(params_path, "r") as f:
    params = json.load(f)

selected_bands = params.get("bands", [])
selected_detectors = params.get("detectors", [])

print(f"Bands from params.json: {selected_bands}")
print(f"Detectors from params.json: {selected_detectors}")

# =====================================================
# Locate VRTs
# =====================================================
vrt_list = glob.glob(
    os.path.join(PATH_L1B_DATA, "DATASTRIP", "S2*", "GEO_DATA", "*.vrt")
)

if not vrt_list:
    raise RuntimeError("No VRT files found in GEO_DATA")

print(f"Total VRT files found: {len(vrt_list)}")

# =====================================================
# Filter VRTs based on params.json
# =====================================================
vrt_names = []
vrt_pattern = r"_D(\d+)_B(\d{1,2}[A]?)$"

for vrt_path in vrt_list:
    vrt_name = os.path.splitext(os.path.basename(vrt_path))[0]
    match = re.search(vrt_pattern, vrt_name)
    
    if match:
        detector = match.group(1)
        band = f"B{match.group(2)}"
        
        # Check if this VRT matches selected bands and detectors
        if detector in selected_detectors and band in selected_bands:
            vrt_names.append(vrt_name)
            print(f"  ✓ Keep: {vrt_name} (detector={detector}, band={band})")
    else:
        print(f"  Warning: Could not parse detector/band from {vrt_name}")

print(f"\nFiltered VRTs to process: {len(vrt_names)}")
print(f"VRT list: {vrt_names}")

# =====================================================
# Output directories
# =====================================================
os.makedirs(os.path.join(OUTPUT_FOLDER, "GDAL_OUTPUT_ORTHO"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_FOLDER, "GDAL_OUTPUT_MOSAIC"), exist_ok=True)

print("Output folder: ", OUTPUT_FOLDER)

# =====================================================
# Build GDAL docker
# =====================================================
notebook_dir = os.path.dirname(os.getcwd())
dockerfile_dir = os.path.abspath(os.path.join(
    notebook_dir,
    "src",
    "gdal-latest"
))

print("\n=== BUILDING GDAL LATEST CONTAINER ===\n")
cmd_build = [
    "docker", "build",
    "--platform=linux/amd64",
    "-t", "gdal-latest",
    dockerfile_dir
]

print("Command:", " ".join(cmd_build), "\n")
subprocess.run(cmd_build, check=True)
print("GDAL image built successfully.\n")

# =====================================================
# Generate gdal_ortho.sh
# =====================================================
gdal_script_path = os.path.join(WORKDIR, "src", "gdal_ortho.sh")
os.makedirs(os.path.dirname(gdal_script_path), exist_ok=True)

# Verify we have VRTs to process
if len(vrt_names) == 0:
    raise RuntimeError(f"No VRT files match the selected bands {selected_bands} and detectors {selected_detectors}")

print(f"\n=== GENERATING GDAL SCRIPT ===")
print(f"VRTs to process: {len(vrt_names)}")
for v in vrt_names:
    print(f"  - {v}")

vrt_array = " ".join([f'"{v}"' for v in vrt_names])

# Band resolution in meters for resampling
# Note: This is different from STEPS which is the grid step in pixels
# STEPS is used for grid generation, while these values are for resampling resolution
resolution_10m = 10  # 10 meters for 10m bands
resolution_20m = 20  # 20 meters for 20m bands
resolution_60m = 60  # 60 meters for 60m bands

with open(gdal_script_path, "w") as f:
    f.write(f"""#!/bin/bash
set +e

cd /data/L1B

OUT_ORTHO="/output/GDAL_OUTPUT_ORTHO"
OUT_MOSAIC="/output/GDAL_OUTPUT_MOSAIC"

mkdir -p "$OUT_ORTHO"
mkdir -p "$OUT_MOSAIC"

XML="{xml_docker}"

# =====================================================
# Function to get band resolution in meters 
# Note: This returns the resampling resolution in meters, not the grid step in pixels
# =====================================================
get_band_resolution() {{
    local vrt_name="$1"
    # Extract band from VRT name (e.g., ..._D09_B01 -> B01)
    local band=$(echo "$vrt_name" | grep -oE '_B[0-9]{{1,2}}[A]?' | sed 's/_//')
    
    case "$band" in
        B02|B03|B04|B08)
            echo {resolution_10m}  # 10m bands -> 10 meters resolution
            ;;
        B05|B06|B07|B8A|B11|B12)
            echo {resolution_20m}  # 20m bands -> 20 meters resolution
            ;;
        B01|B09|B10)
            echo {resolution_60m}  # 60m bands -> 60 meters resolution
            ;;
        *)
            echo {resolution_10m}  # Default to 10 meters
            ;;
    esac
}}

echo "=== ORTHORECTIFICATION ==="

for VRT in {vrt_array}; do
    OUT="$OUT_ORTHO/${{VRT}}_ortho.tif"
    
    echo "----------------------------------------"
    echo "Processing VRT: $VRT"
    echo "----------------------------------------"

    rm -f "$OUT"
    
    # Get resolution for this band (Issue #61)
    RESOLUTION=$(get_band_resolution "$VRT")
    echo "Band resolution: $RESOLUTION m"

    # Orthorectification with explicit resolution and nodata
    gdalwarp \\
        SENTINEL2_L1B_WITH_GEOLOC:./$XML:$VRT \\
        "$OUT" \\
        -t_srs EPSG:{UTM_EPSG} \\
        -te {LOCATION["ul_x"]} {LOCATION["lr_y"]} {LOCATION["lr_x"]} {LOCATION["ul_y"]} \\
        -r cubic \\
        -tr $RESOLUTION -$RESOLUTION \\
        -dstnodata 0 \\
        -co COMPRESS=LZW \\
        -co TILED=YES \\
        -overwrite
done

echo ""
echo "=== MOSAIC GENERATION ==="
echo ""

for BAND in {" ".join(selected_bands)}; do
    echo "----------------------------------------"
    echo "Creating mosaic for band: $BAND"
    echo "----------------------------------------"
    
    INPUT_FILES=($OUT_ORTHO/*_${{BAND}}_ortho.tif)

    if [ ${{#INPUT_FILES[@]}} -eq 0 ]; then
        echo "No ortho images found for band $BAND"
        continue
    fi
    OUTPUT="$OUT_MOSAIC/ORTHO_mosaic_${{BAND}}.tif"
    
    # Use gdal_merge.py instead of gdalwarp to avoid double resampling 
    # All ortho images should have the same resolution and geometry
    # No resampling needed, just assembly
    gdal_merge.py \\
        -o "$OUTPUT" \\
        -of GTiff \\
        -co COMPRESS=LZW \\
        -co TILED=YES \\
        -ot UInt16 \\
        -n 0 \\
        -a_nodata 0 \\
        "${{INPUT_FILES[@]}}"

    echo " Mosaic written -> $OUTPUT"
    echo ""
done

echo "=== GDAL processing complete ==="
""")

os.chmod(gdal_script_path, 0o755)

# =====================================================
# Run GDAL docker
# =====================================================
print("\n=== RUNNING GDAL PROCESSING ===\n")

cmd_run = [
    "docker", "run",
    "--rm",
    "-v", f"{PATH_L1B_DATA}:/data/L1B",
    "-v", f"{WORKDIR}:/workspace",
    "-v", f"{OUTPUT_FOLDER}:/output",
    "gdal-latest",
    "/workspace/src/gdal_ortho.sh"
]

print("Command:", " ".join(cmd_run), "\n")
subprocess.run(cmd_run, check=True)
print("\nGDAL ortho + mosaic complete.\n")
# =====================================================
# Cleanup docker image
# =====================================================

if REMOVE_DOCKER_IMAGE:
    print("Removing gdal-latest image...\n")
    subprocess.run(["docker", "rmi", "-f", "gdal-latest"], check=True)
    print("GDAL image removed.\n")
else:
    print("Docker images kept (faster next run, ~4 GB disk usage).\n")